# 08 · Draft v3 re-analysis (I9, I1, I2, R-3, Table S5)

Open it in Colab and run the setup cell first: it fetches the repository, the OULAD files and the corpus by itself.
It reads `oulad_full_corpus.csv` (written by 01) and writes new CSV files only. It overwrites nothing.

| Part | Supervisor item | What it gives the paper |
|---|---|---|
| A | D-1 / I9 | One declared seed list for the corpus **and** the six courses; Tables 6–7, §5.5, §6.4, abstract, highlight 4 |
| B | R-2 / I1 | Table 9: reported versus fold-internal winsorisation and category levels |
| C | R-1 / I2 | Strength of an unmeasured confounder relative to course identity (Chernozhukov et al., 2026; DoWhy 0.14) |
| D | R-3 | Corpus mean of the outcome (−6.0223) |
| E | P-3 | Table S5, covariate balance, retained versus excluded |

**Before running:** set `SEEDS` in the next code cell. Use the list you declared for the I9 decision.
Copy every printed block back for the manuscript.

## Setup: repository, OULAD files and corpus
Works in a fresh Colab session. It clones the repository, puts the seven OULAD files in `raw/`, and runs
notebook 01 if the corpus file `oulad_full_corpus.csv` is not there yet (about 5–10 minutes).

**Set `DRIVE_RAW`** to the Google Drive folder that holds the seven OULAD CSV files (recommended: `studentVle.csv` is
about 450 MB, too large to upload by hand reliably). Leave it empty to get an upload button instead; a single .zip of the seven files is fine (it is unpacked).

In [2]:
DRIVE_RAW = ''   # optional: path to a mounted folder holding the 7 OULAD csv files

import os, shutil, subprocess
from pathlib import Path
REPO_URL = 'https://github.com/cuasalanonymous-commits/CausalMedia'
NEED = ['studentInfo.csv', 'studentRegistration.csv', 'studentAssessment.csv', 'studentVle.csv',
        'assessments.csv', 'vle.csv', 'courses.csv']

# 1. find the repository, or clone it
root = next((p for p in [Path.cwd(), *Path.cwd().parents, Path('/content/CausalMedia')]
             if (p / '01_corpus').is_dir()), None)
if root is None:
    subprocess.run(['git', 'clone', '-q', REPO_URL, '/content/CausalMedia'], check=True)
    root = Path('/content/CausalMedia')
os.chdir(root)
BASE = str(root)
print('BASE =', BASE)

# 2. the seven OULAD files in raw/
raw = root / 'raw'; raw.mkdir(exist_ok=True)
missing = [f for f in NEED if not (raw / f).exists()]
if missing and DRIVE_RAW:
    from google.colab import drive
    drive.mount('/content/drive')
    for f in missing:
        shutil.copy(os.path.join(DRIVE_RAW, f), raw / f)
    missing = [f for f in NEED if not (raw / f).exists()]
if missing:
    print('Upload these files now:', missing)
    from google.colab import files
    for name, data in files.upload().items():
        (raw / name).write_bytes(data)
    missing = [f for f in NEED if not (raw / f).exists()]
# a zip (e.g. raw.zip or anonymisedData.zip) is unpacked, and files in sub-folders are moved up into raw/
import zipfile
for z in list(raw.glob('*.zip')):
    with zipfile.ZipFile(z) as zf:
        zf.extractall(raw)
for f in NEED:
    if not (raw / f).exists():
        hit = next((q for q in raw.rglob(f) if '__MACOSX' not in q.parts), None)
        if hit is not None:
            shutil.move(str(hit), raw / f)
missing = [f for f in NEED if not (raw / f).exists()]
assert not missing, f'still missing in raw/: {missing}'
print('raw/ complete')

# 3. the corpus file (written by notebook 01 to the repository root)
if not (root / 'oulad_full_corpus.csv').exists():
    print('oulad_full_corpus.csv not found: running notebook 01 ...')
    get_ipython().run_line_magic('run', '01_corpus/01_corpus_construction.ipynb')
    os.chdir(root)
assert (root / 'oulad_full_corpus.csv').exists(), 'notebook 01 did not write oulad_full_corpus.csv'
print('corpus ready:', root / 'oulad_full_corpus.csv')

BASE = /content/CausalMedia
Upload these files now: ['studentInfo.csv', 'studentRegistration.csv', 'studentAssessment.csv', 'studentVle.csv', 'assessments.csv', 'vle.csv', 'courses.csv']


Saving raw.zip to raw.zip
raw/ complete
oulad_full_corpus.csv not found: running notebook 01 ...
OULAD raw directory: /content/CausalMedia/raw

file                            rows  expected  sha256[:8]   status
studentInfo.csv               32,593     32593  7e6f3e47     OK
studentRegistration.csv       32,593     32593  0d326762     OK
studentAssessment.csv        173,912    173912  fd532078     OK
studentVle.csv            10,655,280  10655280  52668253     OK
assessments.csv                  206       206  8cc738fb     OK
vle.csv                        6,364         -  d1b28303     no expectation set
courses.csv                       22        22  4f16eee7     OK
Derived artifacts will be read from / written to: /content/CausalMedia
Total groups: 8459320, groups with >1 row: 1614505
id_student  id_site  date
6516        877011   204     2
                     209     2
                     218     2
            877012   0       4
                     5       2
                     

In [3]:
import zipfile, shutil
for z in list(raw.glob('*.zip')):
    with zipfile.ZipFile(z) as zf:
        zf.extractall(raw)
for f in NEED:
    if not (raw / f).exists():
        hit = next((q for q in raw.rglob(f) if '__MACOSX' not in q.parts), None)
        if hit is not None:
            shutil.move(str(hit), raw / f)
missing = [f for f in NEED if not (raw / f).exists()]
print('still missing:', missing)

still missing: []


In [4]:
!pip install -q econml==0.17.0 dowhy==0.14

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.1/403.1 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.9/245.9 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.3/155.3 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 79.5 MB/s eta 0:00:00


In [5]:
# ===== SET THIS BEFORE RUNNING =====
# Case A (earlier ten-seed run used a different configuration): [42, 1, 7, 123, 2024]
# Case B (same configuration): the ten seeds of the earlier run, exactly as recorded
SEEDS = [42, 1, 7, 123, 2024]
# ===================================
assert SEEDS[0] == 42, 'keep 42 first: it is the canonical seed'

import time, json, math, gc, warnings
import numpy as np, pandas as pd
import econml, sklearn, dowhy
from econml.dml import CausalForestDML, LinearDML
from econml.grf import CausalForest
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold
warnings.filterwarnings('ignore')
print('econml', econml.__version__, '| dowhy', dowhy.__version__, '| sklearn', sklearn.__version__)
assert econml.__version__.startswith('0.17')

CONF = ['region', 'imd_band', 'highest_education', 'age_band', 'gender',
        'disability', 'code_presentation', 'num_of_prev_attempts',
        'studied_credits', 'code_module']
CAT = [c for c in CONF if c not in ('num_of_prev_attempts', 'studied_credits')]
MODS = ['AAA', 'BBB', 'CCC', 'DDD', 'EEE', 'FFF']
CANON_ATE = 0.004786482048864249

full = pd.read_csv(f'{BASE}/oulad_full_corpus.csv')
full['imd_band'] = full['imd_band'].fillna('Missing')
X = pd.get_dummies(full[CONF], columns=CAT, drop_first=True).astype(float)
T = full['oucontent_clicks'].to_numpy(dtype=float)
Y = full['performance_gain'].to_numpy(dtype=float)
assert X.shape == (17529, 40), X.shape
OUT = f'{BASE}/v3_reanalysis'
os.makedirs(OUT, exist_ok=True)

def make_cf(seed, leaf=50):
    return CausalForestDML(model_y=HistGradientBoostingRegressor(random_state=seed),
                           model_t=HistGradientBoostingRegressor(random_state=seed),
                           discrete_treatment=False, honest=True, min_samples_leaf=leaf,
                           n_estimators=500, cv=5, random_state=seed)
print('n =', len(full), '| seeds =', SEEDS)

econml 0.17.0 | dowhy 0.14 | sklearn 1.6.1
n = 17529 | seeds = [42, 1, 7, 123, 2024]


## Part A · I9: one seed list for the corpus and every course
Same configuration as `04_estimation.ipynb` (cell "DDD DIAGNOSTIC", part 3) and `04b_stability.ipynb` (Part B):
leaf 50, 500 trees, cv = 5, HistGradientBoosting nuisances. Seed 42 must reproduce the canonical values, or the cell stops.
Runtime: about 2 minutes per seed on Colab CPU.

In [6]:
corpus_rows, course_cols, t0 = [], {}, time.time()
ccc = (full['code_module'] == 'CCC').to_numpy()
for s in SEEDS:
    cf = make_cf(s); cf.fit(Y, T, X=X)
    lo, hi = [float(np.ravel(v)[0]) for v in cf.ate_interval(X, alpha=0.05)]
    corpus_rows.append({'seed': s, 'ate': float(cf.ate(X)), 'ci_low': lo, 'ci_high': hi})
    eff = pd.Series(cf.effect(X), index=full.index)
    course_cols[s] = eff.groupby(full['code_module']).mean().reindex(MODS)
    if s == 42:
        clo, chi = [float(np.ravel(v)[0]) for v in cf.ate_interval(X[ccc], alpha=0.05)]
        half_width = (chi - clo) / 2
    print(f'seed {s:>5}: corpus ATE {corpus_rows[-1]["ate"]:.6f}  CCC {course_cols[s]["CCC"]:.6f}  [{(time.time()-t0)/60:.1f} min]', flush=True)
    del cf; gc.collect()

corpus = pd.DataFrame(corpus_rows)
courses = pd.DataFrame(course_cols)
# reproduction check against the saved outputs of notebooks 04 / 04b
assert abs(corpus.loc[0, 'ate'] - CANON_ATE) < 1e-6, 'seed 42 does not reproduce 0.004786: stop'
assert abs(courses.loc['CCC', 42] - 0.000129) < 5e-7, 'seed 42 CCC does not reproduce 0.000129: stop'
corpus.to_csv(f'{OUT}/A_corpus_seeds.csv', index=False)
courses.to_csv(f'{OUT}/A_course_seeds.csv')

pd.set_option('display.width', 200)
print('\nCORPUS, per seed'); print(corpus.to_string(index=False, float_format=lambda v: f'{v:.6f}'))
print(f'corpus mean {corpus.ate.mean():.6f}  SD {corpus.ate.std(ddof=1):.6f}  (paper: 0.004788 ± 0.000057, five seeds)')

tab6 = pd.DataFrame({'mean': courses.mean(axis=1), 'sd': courses.std(axis=1, ddof=1)})
tab6['cv_pct'] = 100 * tab6['sd'] / tab6['mean'].abs()
tab6['n'] = full['code_module'].value_counts().reindex(MODS)
med = full.groupby('code_module')['oucontent_clicks'].median().reindex(MODS)
tab6['median_access'] = med
tab6['practical_marks'] = tab6['mean'] * med
print('\nTABLE 6, per course (mean and SD across seeds)'); print(courses.to_string(float_format=lambda v: f'{v:.6f}'))
print(); print(tab6.to_string(float_format=lambda v: f'{v:.6f}'))

seed    42: corpus ATE 0.004786  CCC 0.000129  [0.7 min]
seed     1: corpus ATE 0.004857  CCC 0.000568  [1.0 min]
seed     7: corpus ATE 0.004793  CCC 0.000358  [1.5 min]
seed   123: corpus ATE 0.004699  CCC 0.000616  [1.9 min]
seed  2024: corpus ATE 0.004803  CCC 0.000116  [2.3 min]

CORPUS, per seed
 seed      ate   ci_low  ci_high
   42 0.004786 0.002576 0.006997
    1 0.004857 0.002619 0.007094
    7 0.004793 0.002502 0.007084
  123 0.004699 0.002518 0.006881
 2024 0.004803 0.002712 0.006894
corpus mean 0.004788  SD 0.000057  (paper: 0.004788 ± 0.000057, five seeds)

TABLE 6, per course (mean and SD across seeds)
                42       1        7        123      2024
code_module                                             
AAA         0.004476 0.004443 0.004392 0.004276 0.004499
BBB         0.005054 0.004858 0.004936 0.004754 0.005050
CCC         0.000129 0.000568 0.000358 0.000616 0.000116
DDD         0.004694 0.004840 0.004595 0.004407 0.004661
EEE         0.005500 0.005639 0.0

In [7]:
# Table 7: required n, same rule as notebook 06 (B1), from unrounded values
n_ccc = int(ccc.sum()); FULL_RELEASE = 32593
others = tab6['mean'].drop('CCC')
targets = [('smallest other-course estimate', others.min()),
           ('largest CCC value observed', courses.loc['CCC'].max()),
           (f'CCC {len(SEEDS)}-seed mean', tab6.loc['CCC', 'mean'])]
rows = []
for label, eff in targets:
    n_req = math.ceil(n_ccc * (half_width / eff) ** 2)
    rows.append({'to resolve': label, 'effect': eff, 'required_n': n_req,
                 'vs actual n': 'Achieved' if n_req <= n_ccc else f'{n_req / n_ccc:.1f}x short',
                 'vs full release': f'{n_req / FULL_RELEASE:.1f}x'})
tab7 = pd.DataFrame(rows); tab7.to_csv(f'{OUT}/A_table7.csv', index=False)
print(f'CCC n = {n_ccc:,}   95% half-width (seed 42) = {half_width:.6f}   (paper 0.003332)')
print(tab7.to_string(index=False, float_format=lambda v: f'{v:.6f}'))
print(f'\nhalf-width / CCC mean = {half_width / tab6.loc["CCC", "mean"]:.2f}   (paper: "nearly eight times")')
print(f'CCC per-seed range: {courses.loc["CCC"].min():.6f} to {courses.loc["CCC"].max():.6f}  (paper: 0.000129 to 0.000616)')
print(f'Largest CCC value is still 0.000616? {abs(courses.loc["CCC"].max() - 0.000616) < 5e-7}')

CCC n = 2,114   95% half-width (seed 42) = 0.003332   (paper 0.003332)
                    to resolve   effect  required_n vs actual n vs full release
smallest other-course estimate 0.004417        1203    Achieved            0.0x
    largest CCC value observed 0.000616       61771 29.2x short            1.9x
               CCC 5-seed mean 0.000358      183402 86.8x short            5.6x

half-width / CCC mean = 9.31   (paper: "nearly eight times")
CCC per-seed range: 0.000116 to 0.000616  (paper: 0.000129 to 0.000616)
Largest CCC value is still 0.000616? True


## Part B · I1: fold-internal winsorisation and category levels (Table 9)
The reported pipeline set the outcome cut-points (1st and 99th percentiles) and the category levels once, before cross-fitting.
Here both are computed **inside each training fold** and applied to that fold's held-out part. The nuisance models are fitted by hand
(same learners, same 5 folds), then the same honest causal forest is fitted on the residuals (this is CausalForestDML's final stage).
Row 1 runs the identical hand-built pipeline with the reported global cut-points, so the two rows differ **only** in the preprocessing.
The average-effect interval uses the same rule as EconML's `ate_interval` for a continuous treatment (square root of the mean individual variance).

In [8]:
assert 'performance_gain_raw' in full.columns, 'corpus file has no raw outcome column: re-run notebook 01'
Y_raw = full['performance_gain_raw'].to_numpy(dtype=float)
SD_T = float(np.std(T, ddof=1))   # 781.7 in the paper

def encode(train_df, test_df):
    a = pd.get_dummies(train_df[CONF], columns=CAT, drop_first=True).astype(float)
    b = pd.get_dummies(test_df[CONF], columns=CAT).astype(float).reindex(columns=a.columns, fill_value=0.0)
    return a, b

def residualise(fold_internal, seed=42):
    ry, rt = np.zeros(len(full)), np.zeros(len(full))
    cuts = []
    for tr, te in KFold(5, shuffle=True, random_state=seed).split(full):
        if fold_internal:
            lo, hi = np.quantile(Y_raw[tr], [0.01, 0.99])
            y_tr, y_te = np.clip(Y_raw[tr], lo, hi), np.clip(Y_raw[te], lo, hi)
            x_tr, x_te = encode(full.iloc[tr], full.iloc[te])
        else:
            lo, hi = float(Y.min()), float(Y.max())
            y_tr, y_te = Y[tr], Y[te]
            x_tr, x_te = X.iloc[tr], X.iloc[te]
        cuts.append((lo, hi))
        my = HistGradientBoostingRegressor(random_state=seed).fit(x_tr, y_tr)
        mt = HistGradientBoostingRegressor(random_state=seed).fit(x_tr, T[tr])
        ry[te] = y_te - my.predict(x_te)
        rt[te] = T[te] - mt.predict(x_te)
    return ry, rt, cuts

def final_stage(ry, rt, seed=42):
    f = CausalForest(n_estimators=500, min_samples_leaf=50, honest=True, inference=True,
                     criterion='mse', random_state=seed)
    f.fit(X.to_numpy(), rt.reshape(-1, 1), ry)
    pt, var = f.predict_and_var(X.to_numpy())
    pt, var = np.ravel(pt), np.ravel(var)
    ate, se = pt.mean(), np.sqrt(var.mean())
    return ate, ate - 1.96 * se, ate + 1.96 * se

rows = []
for label, fi in [('Cut-points and levels set once (reported)', False),
                  ('Cut-points and levels set inside each fold', True)]:
    ry, rt, cuts = residualise(fi)
    ate, lo, hi = final_stage(ry, rt)
    rows.append({'preprocessing': label, 'estimate': ate, 'ci_low': lo, 'ci_high': hi,
                 'marks_per_SD': ate * SD_T,
                 'fold cut-points': '; '.join(f'[{a:.2f}, {b:.2f}]' for a, b in cuts)})
    print(f'{label}: {ate:.6f}  [{lo:.6f}, {hi:.6f}]  {ate * SD_T:.2f} marks per SD', flush=True)
tab9 = pd.DataFrame(rows); tab9.to_csv(f'{OUT}/B_table9_fold_internal.csv', index=False)
print(); print(tab9.to_string(index=False, float_format=lambda v: f'{v:.6f}'))
print(f'\nSD of access = {SD_T:.1f}   canonical EconML fit: 0.004786 [0.002576, 0.006997], 3.74 marks per SD')
print(f'change from row 1 to row 2: {tab9.estimate[1] - tab9.estimate[0]:+.6f} '
      f'({100 * (tab9.estimate[1] / tab9.estimate[0] - 1):+.2f}%)')

Cut-points and levels set once (reported): 0.004786  [0.002576, 0.006997]  3.74 marks per SD
Cut-points and levels set inside each fold: 0.004824  [0.002629, 0.007019]  3.77 marks per SD

                             preprocessing  estimate   ci_low  ci_high  marks_per_SD                                                                     fold cut-points
 Cut-points and levels set once (reported)  0.004786 0.002576 0.006997      3.741734 [-61.38, 40.00]; [-61.38, 40.00]; [-61.38, 40.00]; [-61.38, 40.00]; [-61.38, 40.00]
Cut-points and levels set inside each fold  0.004824 0.002629 0.007019      3.771059 [-62.00, 38.00]; [-62.00, 38.78]; [-61.00, 39.00]; [-61.00, 39.00]; [-62.00, 39.00]

SD of access = 781.7   canonical EconML fit: 0.004786 [0.002576, 0.006997], 3.74 marks per SD
change from row 1 to row 2: +0.000038 (+0.78%)


## Part C · I2: unmeasured-confounder strength relative to course identity
DoWhy `add_unobserved_common_cause` with `simulation_method="non-parametric-partial-R2"` and
`benchmark_common_causes` = the five course-identity columns (Chernozhukov et al., 2026).
It first tries the causal forest. If DoWhy refuses a continuous treatment with the forest, it falls back to the
linear DML fit and prints **FALLBACK**: then Section 4.9 must say the bound was computed on the linear DML fit.
The loop scales the confounder to k times the strength of course identity and finds where the interval reaches zero.
This is the slowest part (each k refits the nuisance models). Results are saved after every k, so a disconnect loses nothing.

In [9]:
import networkx as nx
from dowhy import CausalModel

COURSE_COLS = [c for c in X.columns if c.startswith('code_module_')]
print('benchmark columns:', COURSE_COLS)
df = X.copy(); df['T'] = T; df['Y'] = Y
G = nx.DiGraph()
for c in X.columns:
    G.add_edge(c, 'T'); G.add_edge(c, 'Y')
G.add_edge('T', 'Y')
EM = list(X.columns)
model_cf = CausalModel(data=df, treatment='T', outcome='Y', graph=G, effect_modifiers=EM)   # as in 05b
model_lin = CausalModel(data=df, treatment='T', outcome='Y', graph=G)                      # partially linear
estimand = model_cf.identify_effect(proceed_when_unidentifiable=True)

def hgb(): return HistGradientBoostingRegressor(random_state=42)
FIT = None
try:
    model = model_cf
    est = model.estimate_effect(estimand, method_name='backdoor.econml.dml.CausalForestDML',
        effect_modifiers=EM,
        method_params={'init_params': dict(model_y=hgb(), model_t=hgb(), discrete_treatment=False,
                       honest=True, min_samples_leaf=50, n_estimators=500, cv=5, random_state=42),
                       'fit_params': {}})
    print('forest estimate via DoWhy:', float(np.mean(np.ravel(est.value))), '(canonical 0.004786)')
    probe = model.refute_estimate(estimand, est, method_name='add_unobserved_common_cause',
        simulation_method='non-parametric-partial-R2', benchmark_common_causes=COURSE_COLS,
        effect_fraction_on_treatment=1.0, effect_fraction_on_outcome=1.0, plot_estimate=False)
    FIT = 'causal forest'
except Exception as e:
    print('Forest refused by the refuter:', type(e).__name__, str(e)[:300])
    print('FALLBACK: running the bound on the linear DML fit (say so in Section 4.9)')
    model = model_lin
    estimand = model.identify_effect(proceed_when_unidentifiable=True)
    est = model.estimate_effect(estimand, method_name='backdoor.econml.dml.LinearDML',
        method_params={'init_params': dict(model_y=hgb(), model_t=hgb(), discrete_treatment=False,
                       cv=5, random_state=42), 'fit_params': {}})
    probe = model.refute_estimate(estimand, est, method_name='add_unobserved_common_cause',
        simulation_method='non-parametric-partial-R2', benchmark_common_causes=COURSE_COLS,
        effect_fraction_on_treatment=1.0, effect_fraction_on_outcome=1.0, plot_estimate=False)
    FIT = 'linear DML'
print('\nFIT USED:', FIT, '| estimate', est.value)
print(probe)
for a in ['RV', 'RV_alpha', 'r2tu_w', 'r2yu_tw', 'benchmarking']:
    if hasattr(probe, a): print(a, '=', getattr(probe, a))
if hasattr(probe, 'results'): print(probe.results)

benchmark columns: ['code_module_BBB', 'code_module_CCC', 'code_module_DDD', 'code_module_EEE', 'code_module_FFF']


forest estimate via DoWhy: 0.004786482048864249 (canonical 0.004786)

FIT USED: causal forest | estimate 0.004786482048864249
Sensitivity Analysis to Unobserved Confounding using partial R^2 parameterization

Original Effect Estimate : 0.004786482048864249
Robustness Value : 0.03

Robustness Value (alpha=0.05) : 0.01

Interpretation of results :
Any confounder explaining less than 3.0% percent of the residual variance of both the treatment and the outcome would not be strong enough to explain away the observed effect i.e bring down the estimate to 0 

For a significance level of 5.0%, any confounder explaining more than 1.0% percent of the residual variance of both the treatment and the outcome would be strong enough to make the estimated effect not 'statistically significant'


RV = 0.03
RV_alpha = 0.01
r2tu_w = 0.9926970619792086
r2yu_tw = 0.06213209225081458
benchmarking = True
     r2tu_w   r2yu_tw  short estimate      bias  lower_ate_bound  upper_ate_bound  lower_confidence_bound 

In [10]:
def bounds(ref):
    # pull the lower and upper confidence bounds out of the refuter result, whatever the field names
    r = getattr(ref, 'results', None)
    if isinstance(r, pd.DataFrame) and len(r):
        row = r.iloc[-1]
        lo = next((row[c] for c in r.columns if 'lower' in c and 'confidence' in c), None)
        hi = next((row[c] for c in r.columns if 'upper' in c and 'confidence' in c), None)
        if lo is None: lo = next((row[c] for c in r.columns if 'lower' in c), np.nan)
        if hi is None: hi = next((row[c] for c in r.columns if 'upper' in c), np.nan)
        return float(lo), float(hi)
    return np.nan, np.nan

K_GRID = [0.05, 0.1, 0.2, 0.3, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0]
ck = f'{OUT}/C_sensitivity_grid.csv'
done = pd.read_csv(ck) if os.path.exists(ck) else pd.DataFrame(columns=['k', 'lower', 'upper'])
for k in K_GRID:
    if k in set(done['k']): continue
    try:
        ref = model.refute_estimate(estimand, est, method_name='add_unobserved_common_cause',
            simulation_method='non-parametric-partial-R2', benchmark_common_causes=COURSE_COLS,
            effect_fraction_on_treatment=k, effect_fraction_on_outcome=k, plot_estimate=False)
        lo, hi = bounds(ref)
    except Exception as e:
        print(f'k={k}: failed ({type(e).__name__}: {str(e)[:200]})'); lo = hi = np.nan
    done = pd.concat([done, pd.DataFrame([{'k': k, 'lower': lo, 'upper': hi}])], ignore_index=True)
    done.to_csv(ck, index=False)
    print(f'k = {k:<5} bias-adjusted 95% bounds [{lo:.6f}, {hi:.6f}]', flush=True)

g = done.sort_values('k').reset_index(drop=True)
cross = g[g['lower'] <= 0]
print('\nFIT USED:', FIT)
if len(cross):
    i = cross.index[0]
    if i == 0:
        print(f'I2 VALUE: the interval already reaches zero at k = {g.k[0]} (below the grid)')
    else:
        k0, k1, l0, l1 = g.k[i - 1], g.k[i], g.lower[i - 1], g.lower[i]
        kstar = k0 + (0 - l0) * (k1 - k0) / (l1 - l0)
        print(f'I2 VALUE: the interval reaches zero between k = {k0} and {k1}; interpolated k* = {kstar:.2f} times course identity')
else:
    print(f'I2 VALUE: the interval stays above zero up to k = {g.k.max()} times course identity')

k = 0.05  bias-adjusted 95% bounds [-0.001090, 0.010680]
k = 0.1   bias-adjusted 95% bounds [-0.004323, 0.013941]
k = 0.2   bias-adjusted 95% bounds [-0.010917, 0.020547]


KeyboardInterrupt: 

**Run note.** The grid was stopped after k = 0.2 (KeyboardInterrupt above): the bias-adjusted interval already includes zero at k = 0.05, the smallest value in the grid, so larger k cannot change the reported result ("under 0.05 times the strength of course identity"). The robustness values in the cell before (0.03 and 0.01) are complete.

## Part D · R-3: corpus mean of the outcome
(Also paste the same line at the end of `06_verification/06_reported_numbers.ipynb`, as the supervisor asked.)

In [13]:
print(f"corpus mean of performance_gain = {full['performance_gain'].mean():.4f}   (paper: -6.0223)")

corpus mean of performance_gain = -6.0223   (paper: -6.0223)


## Part E · Table S5: covariate balance, retained versus excluded
Same comparison as `01_corpus_construction.ipynb` (PART 2, GGG removed): 17,529 retained against 12,530 excluded.

In [14]:
KEY = ['id_student', 'code_module', 'code_presentation']
BAL = ['gender', 'region', 'highest_education', 'imd_band', 'age_band',
       'num_of_prev_attempts', 'studied_credits', 'disability', 'code_module', 'code_presentation']
si = pd.read_csv(f'{BASE}/raw/studentInfo.csv')[KEY + [c for c in BAL if c not in KEY]].drop_duplicates(KEY)
si = si[si['code_module'] != 'GGG']
kept = set(map(tuple, full[KEY].drop_duplicates().values))
si['_kept'] = [tuple(k) in kept for k in si[KEY].values]
ret, exc = si[si['_kept']], si[~si['_kept']]

def smd(a, b):
    sp = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
    return 0.0 if sp == 0 or np.isnan(sp) else (a.mean() - b.mean()) / sp
rows = []
for c in BAL:
    if pd.api.types.is_numeric_dtype(si[c]):
        rows.append({'variable': c, 'level': '(mean)', 'retained': ret[c].mean(), 'excluded': exc[c].mean(),
                     'SMD': smd(ret[c].dropna(), exc[c].dropna())})
    else:
        for lv in sorted(set(si[c].dropna())):
            rows.append({'variable': c, 'level': lv, 'retained': (ret[c] == lv).mean(), 'excluded': (exc[c] == lv).mean(),
                         'SMD': smd((ret[c] == lv).astype(float), (exc[c] == lv).astype(float))})
s5 = pd.DataFrame(rows)
s5['|SMD| > 0.1'] = np.where(s5['SMD'].abs() > 0.1, 'yes', '')
s5.to_csv(f'{OUT}/E_table_S5_balance.csv', index=False)
print(f'retained = {len(ret):,}   excluded = {len(exc):,}')
print(f"{(s5['SMD'].abs() > 0.1).sum()} of {len(s5)} comparisons exceed |SMD| = 0.1   (paper: 15 of 47)")
print(s5.to_string(index=False, float_format=lambda v: f'{v:.4f}'))

retained = 17,529   excluded = 12,530
15 of 47 comparisons exceed |SMD| = 0.1   (paper: 15 of 47)
            variable                       level  retained  excluded     SMD |SMD| > 0.1
              gender                           F    0.4036    0.4470 -0.0880            
              gender                           M    0.5964    0.5530  0.0880            
              region         East Anglian Region    0.1015    0.0967  0.0161            
              region        East Midlands Region    0.0678    0.0752 -0.0285            
              region                     Ireland    0.0465    0.0287  0.0935            
              region               London Region    0.0892    0.1042 -0.0508            
              region                North Region    0.0598    0.0553  0.0195            
              region        North Western Region    0.0791    0.1001 -0.0736            
              region                    Scotland    0.1195    0.1023  0.0548            
            

## Done
Download the folder `v3_reanalysis/` (five CSV files) and paste every printed block above back to the writing team.